# Extractive Question Answering: SQuAD, Span Prediction and Fine-Tuning

## Overview
**Extractive QA**: given a passage (context) and a question, find the exact text span in the passage that answers it. The model does not write text; it points to a start and an end token.

In this notebook you will:
1. **Explore SQuAD** and implement the **Exact Match (EM)** and **F1** metrics.
2. **Run a pre-trained QA model** (zero-shot), including a question that has *no answer* in the passage (SQuAD 2.0).
3. **Visualize start/end probabilities** per token to see how the span is chosen.
4. **Align answers to tokens**: map the character span of an answer to token positions (the labels used in training).
5. **Fine-tune a small model on real SQuAD data** and compare it with the pre-trained model using EM and F1.

In [ ]:
# The Hugging Face Trainer needs `accelerate`. Install once, then restart the kernel if it was missing.
%pip install -q "accelerate>=1.1.0" datasets

In [ ]:
import re, string
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (AutoModelForQuestionAnswering, AutoTokenizer, Trainer,
                          TrainingArguments, default_data_collator)

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

---
## 1. The SQuAD Benchmark and Its Metrics

SQuAD 1.1 has 100,000+ questions written by crowdworkers about Wikipedia paragraphs. Each answer is a span of the paragraph, stored as text plus the **character position** where it starts (`answer_start`). SQuAD 2.0 adds 50,000+ unanswerable questions.

In [ ]:
squad = load_dataset("squad")
print(squad)
ex = squad["validation"][0]
print("\nQuestion:", ex["question"])
print("Context :", ex["context"][:300], "...")
print("Answers :", ex["answers"])

### Metrics
- **Exact Match (EM)**: 1 if the predicted string equals a gold answer after normalization (lowercase, remove punctuation and articles), else 0.
- **F1**: overlap in words between prediction and gold answer: $F_1 = \frac{2PR}{P+R}$, so partial answers earn partial credit.

When there are several gold answers, we take the best score.

In [ ]:
def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def f1_score(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)

gold = "the River Thames"
for pred in ["River Thames", "the River Thames in England", "London", "Thames"]:
    print(f"{pred!r:32} EM={exact_match(pred, gold):.0f}  F1={f1_score(pred, gold):.2f}")

---
## 2. Zero-Shot QA with a Pre-Trained Model

`deepset/roberta-base-squad2` was fine-tuned on SQuAD 2.0. We call the model directly (the `question-answering` pipeline was removed in transformers v5).

The model outputs a **start logit** and an **end logit** for every token. The answer is the span (start, end) with the highest combined probability, where `end >= start`. For SQuAD 2.0 the `[CLS]` position (index 0) represents **"no answer"**: if its score beats every real span, the model abstains.

In [ ]:
def extract_answer(model, tok, question, context, max_answer_tokens=30, allow_null=False):
    """Return the best answer span (and the probabilities used to choose it)."""
    enc = tok(question, context, return_tensors="pt", truncation="only_second",
              max_length=384, return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    seq_ids = enc.sequence_ids(0)
    inputs = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**inputs)
    start_p = torch.softmax(out.start_logits[0], dim=-1).cpu()
    end_p = torch.softmax(out.end_logits[0], dim=-1).cpu()

    ctx_idx = [i for i, s in enumerate(seq_ids) if s == 1]        # tokens that belong to the context
    best_score, best_s, best_e = 0.0, ctx_idx[0], ctx_idx[0]
    for s in ctx_idx:
        for e in range(s, min(s + max_answer_tokens, len(seq_ids))):
            if seq_ids[e] != 1:
                break
            score = float(start_p[s] * end_p[e])
            if score > best_score:
                best_score, best_s, best_e = score, s, e
    null_score = float(start_p[0] * end_p[0])                       # [CLS] = "no answer"
    answer = context[offsets[best_s][0]:offsets[best_e][1]]
    if allow_null and null_score > best_score:
        answer = ""
    return {"answer": answer, "score": best_score, "null_score": null_score,
            "ctx_tokens": tok.convert_ids_to_tokens(enc["input_ids"][0])[ctx_idx[0]:ctx_idx[-1] + 1],
            "start_p": start_p[ctx_idx].numpy(), "end_p": end_p[ctx_idx].numpy()}

model_name = "deepset/roberta-base-squad2"
qa_tok = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name).eval().to(device)

context = ("London is the capital and largest city of England and the United Kingdom. "
           "Standing on the River Thames in south-east England, at the head of a 50-mile estuary "
           "down to the North Sea, it has been a major settlement for two millennia.")
questions = ["What river does London stand on?",
             "How long is the estuary down to the North Sea?",
             "What is the population of London?"]      # NOT answerable from this passage

rows = []
for q in questions:
    r = extract_answer(qa_model, qa_tok, q, context, allow_null=True)
    rows.append({"Question": q, "Answer": r["answer"] or "(no answer)",
                 "Span score": round(r["score"], 3), "No-answer score": round(r["null_score"], 3)})
pd.DataFrame(rows)

For the first two questions the span score is far above the no-answer score. For the **population** question the passage never says, so the no-answer score wins and the model abstains instead of inventing a number. Extractive QA cannot hallucinate text that is not in the passage, but it can still pick a wrong span when it should have abstained.

---
## 3. Visualizing Start and End Probabilities

Plot the probability that each context token is the **start** (blue) or the **end** (orange) of the answer.

In [ ]:
def plot_span_probs(question, context):
    r = extract_answer(qa_model, qa_tok, question, context, allow_null=True)
    x = np.arange(len(r["ctx_tokens"]))
    fig, ax = plt.subplots(figsize=(13, 3.2))
    ax.bar(x - 0.2, r["start_p"], width=0.4, label="P(start)", color="#1f77b4")
    ax.bar(x + 0.2, r["end_p"], width=0.4, label="P(end)", color="#ff7f0e")
    ax.set_xticks(x); ax.set_xticklabels(r["ctx_tokens"], rotation=75, fontsize=7)
    ax.set_title(f"{question}   ->   {r['answer'] or '(no answer)'}")
    ax.legend(); plt.tight_layout(); plt.show()

plot_span_probs("What river does London stand on?", context)
plot_span_probs("How long is the estuary down to the North Sea?", context)

The model concentrates start probability on the first word of the answer and end probability on the last word. The predicted span is the pair (start, end) with the highest product, subject to `end >= start`.

---
## 4. From Character Spans to Token Labels

SQuAD stores the answer as a **character offset**, but the model predicts **token positions**. To train, we convert one into the other using the tokenizer's `offset_mapping` (which character range each token covers). This also shows how the question and passage are packed together: `[CLS] question [SEP] passage [SEP]`.

In [ ]:
def char_to_token_positions(offsets, seq_ids, ans_start, ans_end):
    """Token (start, end) positions for a character span, or None if the answer was truncated."""
    ctx = [i for i, s in enumerate(seq_ids) if s == 1]
    cs, ce = ctx[0], ctx[-1]
    if offsets[cs][0] > ans_start or offsets[ce][1] < ans_end:
        return None
    s = cs
    while s <= ce and offsets[s][0] <= ans_start:
        s += 1
    s -= 1
    e = ce
    while e >= cs and offsets[e][1] >= ans_end:
        e -= 1
    e += 1
    return s, e

tok_bert = AutoTokenizer.from_pretrained("distilbert-base-uncased")
ex = squad["train"][3]
a0 = ex["answers"]["answer_start"][0]
a1 = a0 + len(ex["answers"]["text"][0])
enc = tok_bert(ex["question"], ex["context"], truncation="only_second", max_length=384, return_offsets_mapping=True)
s, e = char_to_token_positions(enc["offset_mapping"], enc.sequence_ids(0), a0, a1)
tokens = tok_bert.convert_ids_to_tokens(enc["input_ids"])

print("Question     :", ex["question"])
print("Gold answer  :", ex["answers"]["text"][0], f"(characters {a0} to {a1})")
print("Token labels : start =", s, " end =", e)
print("Tokens in span:", tokens[s:e + 1])
print("Decoded span  :", tok_bert.decode(enc["input_ids"][s:e + 1]))
print("\nFirst tokens of the input:", tokens[:12])

These two integers (`start_positions`, `end_positions`) are the **only labels** QA fine-tuning needs. The loss is cross-entropy over token positions for the start and for the end, averaged.

---
## 5. Fine-Tuning on Real SQuAD Data

We fine-tune `distilbert-base-uncased` (small and fast) on a **subset of SQuAD** and compare three models on 200 validation questions:

1. DistilBERT with a **random QA head** (before training),
2. DistilBERT **fine-tuned here** on a small subset,
3. `roberta-base-squad2`, fine-tuned by others on the full dataset.

Adjust `N_TRAIN` (more data means a better score but a longer run; roughly 5 to 10 minutes on an Apple-silicon GPU or a Colab GPU for 3,000 examples).

In [ ]:
N_TRAIN, N_EVAL = 3000, 200

def prep_train(batch):
    enc = tok_bert(batch["question"], batch["context"], truncation="only_second",
                   max_length=384, padding="max_length", return_offsets_mapping=True)
    starts, ends = [], []
    for i, offsets in enumerate(enc["offset_mapping"]):
        ans = batch["answers"][i]
        a0 = ans["answer_start"][0]
        pos = char_to_token_positions(offsets, enc.sequence_ids(i), a0, a0 + len(ans["text"][0]))
        starts.append(pos[0] if pos else 0)
        ends.append(pos[1] if pos else 0)
    enc["start_positions"], enc["end_positions"] = starts, ends
    enc.pop("offset_mapping")
    return enc

raw_train = squad["train"].shuffle(seed=0).select(range(N_TRAIN))
train_ds = raw_train.map(prep_train, batched=True, remove_columns=raw_train.column_names)
train_ds = train_ds.filter(lambda x: x["start_positions"] > 0)      # drop answers cut off by truncation
print("Training examples:", len(train_ds))

eval_examples = squad["validation"].shuffle(seed=0).select(range(N_EVAL))

def evaluate_squad(model, tok, examples):
    em = f1 = 0.0
    for ex in examples:
        pred = extract_answer(model, tok, ex["question"], ex["context"])["answer"]
        golds = ex["answers"]["text"]
        em += max(exact_match(pred, g) for g in golds)
        f1 += max(f1_score(pred, g) for g in golds)
    return 100 * em / len(examples), 100 * f1 / len(examples)

ft_model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased").to(device)
em0, f10 = evaluate_squad(ft_model.eval(), tok_bert, eval_examples)
print(f"Before training (random QA head): EM={em0:.1f}  F1={f10:.1f}")

In [ ]:
args = TrainingArguments(
    output_dir="./results_squad", num_train_epochs=2, learning_rate=5e-5,
    per_device_train_batch_size=16, save_strategy="no", eval_strategy="no",
    logging_steps=25, report_to="none",
)
trainer = Trainer(model=ft_model, args=args, train_dataset=train_ds,
                  data_collator=default_data_collator, processing_class=tok_bert)
trainer.train()

In [ ]:
em1, f11 = evaluate_squad(ft_model.eval(), tok_bert, eval_examples)
em2, f12 = evaluate_squad(qa_model, qa_tok, eval_examples)

results = pd.DataFrame({
    "Model": ["DistilBERT, random head", f"DistilBERT, fine-tuned on {len(train_ds):,} examples",
              "RoBERTa-base, fine-tuned on full SQuAD 2.0 (zero-shot here)"],
    "EM": [em0, em1, em2], "F1": [f10, f11, f12],
}).round(1)
display(results)
results.set_index("Model")[["EM", "F1"]].plot.barh(figsize=(9, 3), title=f"SQuAD validation ({N_EVAL} questions)")
plt.tight_layout(); plt.show()

In [ ]:
# A few predictions from the model you just fine-tuned
rows = []
for ex in eval_examples.select(range(6)):
    pred = extract_answer(ft_model, tok_bert, ex["question"], ex["context"])["answer"]
    rows.append({"Question": ex["question"], "Gold": ex["answers"]["text"][0], "Predicted": pred,
                 "F1": round(f1_score(pred, ex["answers"]["text"][0]), 2)})
pd.DataFrame(rows)

**Reading the results.** The random head scores close to 0. A few thousand training examples already teach DistilBERT to find spans, but it stays well below the RoBERTa model trained on the full dataset (and with a much larger encoder). Try changing `N_TRAIN` and watch how EM and F1 move: this is the data-versus-quality trade-off you will meet again in fine-tuning (Week 5).

## 6. Summary & Key Takeaways

1. **Extractive vs. generative QA**: the model points to a start and an end token in the passage, so it cannot invent text that is not there. It is limited to substrings.
2. **Span scoring**: each token gets a start logit and an end logit; the answer is the best valid pair with `end >= start`.
3. **No-answer handling (SQuAD 2.0)**: the `[CLS]` position stands for "no answer"; the model abstains when that score beats every span.
4. **Labels come from offsets**: character offsets are converted into token start and end positions with the tokenizer's `offset_mapping`.
5. **Metrics**: Exact Match is strict; F1 gives partial credit for overlapping words.
6. **Modern paradigm**: extractive QA dominated 2018 to 2022; today's RAG systems retrieve passages and let a generative LLM answer, but span extraction is still used for precise, citable answers.